##### transform.ipynb  —  Reusable standardization functions (Q3)

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

In [2]:
def standardize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize every column name in the DataFrame:
        - strip leading/trailing spaces
        - collapse internal multiple spaces
        - lowercase
        - replace spaces with underscores
    e.g. `' Order  ID '`  ->  `'order_id'`

    This is reusable on ANY dataframe.
    """
    clean_cols = {}
    for col in df.columns:
        s = str(col)
        s = s.strip()                     # remove outer blanks
        s = re.sub(r"\s+", " ", s)        # collapse multiple spaces
        s = s.lower()                     # lowercase
        s = s.replace(" ", "_")           # spaces -> underscores
        clean_cols[col] = s
    return df.rename(columns=clean_cols)

In [3]:
def alias_columns(df: pd.DataFrame, aliases: dict) -> pd.DataFrame:
    """
    Merge columns that may use synonyms into one canonical name.
    Eg aliases = { 'transactionid': 'transaction_id',
                   'transaction id': 'transaction_id' }

    Useful when the same concept is named slightly differently
    across the four raw files.
    """
    return df.rename(columns=aliases)

In [4]:
def standardize_dates(df: pd.DataFrame,
                      date_cols: list[str],
                      output_format: str = "%Y-%m-%d") -> pd.DataFrame:
    """
    Convert every value in the given date columns to a single,
    consistent string format (default ISO 'YYYY-MM-DD').

    The function is robust: it tries several common datetime
    formats (dd/mm/yyyy, yyyy-mm-dd, mm/dd/yyyy ...) and falls
    back to pandas' flexible parser.

    Args:
        df: input DataFrame (already column-standardized).
        date_cols: list of column names that hold dates.
        output_format: strftime target format.

    Returns:
        A copy of df with the date columns re-formatted.
    """
    df = df.copy()

    for col in date_cols:
        if col not in df.columns:
            print(f"[warn] date column '{col}' not present — skipped")
            continue

        parsed = pd.to_datetime(df[col], errors="coerce")

        n_bad = parsed.isna().sum()
        if n_bad > 0:
            print(f"[warn] '{col}': {n_bad} unparseable value(s) -> NaT")

        df[col] = parsed

    return df

In [5]:
def clean_whitespace(df: pd.DataFrame, text_cols: list[str]) -> pd.DataFrame:  
    """  
    For every text column: strip leading/trailing whitespace and  
    collapse inner multiple spaces so no accidental blank padding  
    survives. Missing strings become "" to keep comparing easy.  
    """  
    df = df.copy()  
    for col in text_cols:  
        if col not in df.columns:  
            continue  
        df[col] = (df[col]  
                   .astype("string")                 # safe text dtype  
                   .str.strip()                      # remove outer blanks  
                   .str.replace(r"\s+", " ", regex=True))  # collapse inner  
        df[col] = df[col].fillna("")                 # NaN -> empty string  
    return df

In [6]:
def normalize_case(df: pd.DataFrame,
                   text_cols: list[str],
                   style: str = "lower") -> pd.DataFrame:
    """
    Uniform text casing to avoid 'PENDING', 'pending', 'Pending'
    being treated as different values.

    Arg 'style': 'lower' | 'upper' | 'title'
    """
    df = df.copy()
    for col in text_cols:
        if col not in df.columns:
            continue
        if style == "lower":  
            df[col] = df[col].str.lower()
        elif style == "upper":  
            df[col] = df[col].str.upper()
        elif style == "title":  
            df[col] = df[col].str.title()
        else:  
            raise ValueError("style must be 'lower'|'upper'|'title'")
    return df

In [7]:
def strip_non_alpha(df: pd.DataFrame,
                    text_cols: list[str]) -> pd.DataFrame:
    """
    Remove all characters EXCEPT letters, digits and spaces
    from the given text columns.

    Simple and reliable: keeps only [a-zA-Z0-9 ] no matter what,
    then collapses leftover blank spaces and strips edges.
    """
    df = df.copy()
    keep_pattern = "[^a-zA-Z0-9 ]"     # remove everything not allowed

    for col in text_cols:
        if col not in df.columns:
            print(f"[warn] column '{col}' not found — skipped")
            continue
        df[col] = df[col].astype("string")
        df[col] = df[col].str.replace(keep_pattern, "", regex=True)
        df[col] = df[col].str.replace(r"\s+", " ", regex=True).str.strip()

    return df

In [8]:
def coerce_numeric(df: pd.DataFrame,
                   numeric_cols: list[str]) -> pd.DataFrame:
    """
    Force the given columns to a numeric dtype.

    Values like '12', '12.5', '€1,200', '1 200' are all cleaned
    and coerced to a proper number. Anything that truly cannot be
    parsed becomes NaN (missing) so it does not crash downstream.

    Applies the standard pandas int/float promotion: if any value
    in a column is non-integer the whole column becomes float.

    Args:
        df: input DataFrame.
        numeric_cols: columns intended to hold numeric data.

    Returns:
        DataFrame with those columns re-cast to numeric.
    """
    df = df.copy()

    for col in numeric_cols:
        if col not in df.columns:
            print(f"[warn] numeric column '{col}' not found — skipped")
            continue

        # --- 1) make sure it is string text first ---
        s = df[col].astype("string")

        # --- 2) strip thousands separators & currency symbols ---
        s = s.str.replace(r"[\$,]", "", regex=True)     # drop $ and ,
        s = s.str.replace(r"\s+", "", regex=True)       # drop blank spaces
        # (careful: some locales use '.' as thousands sep — skip for now)

        # --- 3) convert to numeric; unparseable -> NaN ---
        df[col] = pd.to_numeric(s, errors="coerce")

    return df

In [9]:
def standardize_currency(df: pd.DataFrame,
                         amount_cols: list[str],
                         decimals: int = 2) -> pd.DataFrame:
    """
    Round monetary amounts to a consistent number of decimals and
    keep them as float. Call AFTER coerce_numeric so the column is
    already numeric.
    """
    df = df.copy()
    for col in amount_cols:
        if col in df.columns:
            df[col] = df[col].round(decimals)
    return df

In [10]:
def normalize_ids(df: pd.DataFrame,
                  id_cols: list[str],
                  prefix: str | None = None,
                  uppercase: bool = True,
                  width: int | None = None,
                  remove_seps: bool = True) -> pd.DataFrame:
    """
    Bring ID-like fields (order_id, product_id, sku, customer_id...)
    to a single, predictable shape so that joins across the raw files
    match cleanly and consistently.

    What this function does, in order:
      1. strips leading/trailing whitespace
      2. (optional) removes hyphen/space/underscore separators so the
         id becomes ONE continuous token, e.g. 'ord-123' -> 'ord123'
      3. (optional) forces uppercase, since most ids are compared
         case-insensitively ('ab12' vs 'AB12')
      4. (optional) left-pads the numeric tail to a fixed width and
         prepends a fixed prefix, e.g. width=5, prefix='C'
             '7'   -> 'C00007'
             'ab1' -> 'Cab001'

    Args:
        df: input DataFrame.
        id_cols: columns that hold identifiers.
        prefix: optional fixed prefix to put in front (e.g. 'ORD').
        uppercase: if True, cast the whole token to upper case.
        width: optional integer; the trailing digits will be zero-padded
               so the whole tail has this many digits.
        remove_seps: if True, drop '-', '_' and spaces from the token.

    Returns:
        A copy of the DataFrame with the id columns re-standardized.
    """
    df = df.copy()

    for col in id_cols:
        if col not in df.columns:
            print(f"[warn] id column '{col}' not found — skipped")
            continue

        s = df[col].astype("string")

        # (1) trim outer whitespace
        s = s.str.strip()

        # guard: leave missing values untouched
        mask_missing = s.isna()

        # (2) optionally collapse separators to build one token
        if remove_seps:
            s = s.str.replace(r"[\s\-_]", "", regex=True)

        # (3) optionally force uppercase
        if uppercase:
            s = s.str.upper()

        # (4) optional zero-padding of the trailing numeric part
        if width is not None:
            # split token into leading letters and trailing digits
            parts = s.str.extract(r"^([A-Za-z]*)(\d*)$")
            letters = parts[0].fillna("")
            digits  = parts[1].fillna("")
            digits_padded = digits.str.zfill(width)       # pad to width
            s = letters + digits_padded

        # (5) optional fixed prefix
        if prefix is not None:
            s = prefix + s

        # restore missing as NaN
        s = s.mask(mask_missing) if hasattr(s, "mask") else s
        df[col] = s

    return df